# Gold Layer: Dimension Customers (Comment-Driven DQ)

**Pattern**: Metadata-Driven Quality Control via Column Comments  
**Sources**: Silver CRM Customers

This notebook enforces quality rules that are defined directly in the **Table Metadata (Comments)**.

In [0]:
# 1. Include the Generic DQ Engine
%run "../utils/utils_quality_control"

### Step 0: Define Metadata (The 'Invisible' Rules)
Run this once to 'Tag' your columns with quality rules. In the future, if you add a new column, you just add a similar comment!

In [0]:
target_table = "workspace.gold.dim_customers"

if spark.catalog.tableExists(target_table):
    spark.sql(f"ALTER TABLE {target_table} CHANGE COLUMN customer_id COMMENT 'DQ: NOT NULL'")
    spark.sql(f"ALTER TABLE {target_table} CHANGE COLUMN first_name COMMENT 'DQ: NOT NULL'")
    print("✓ Column metadata (DQ Tags) updated in Catalog.")
else:
    print("ℹ Table does not exist yet. Run the stream first to create it, then run this cell to add DQ rules.")

In [0]:
from pyspark.sql.functions import col, lit
from delta.tables import DeltaTable

# 2. Configuration
quarantine_table = "workspace.gold.quarantine_customers"
checkpoint_path = "/Volumes/workspace/bronze/raw_sources/checkpoints/gold_dim_customers"

def process_gold_customers(batch_df, batch_id):
    # Check if table exists to decide if we apply DQ
    table_exists = spark.catalog.tableExists(target_table)
    
    if table_exists:
        # A. Apply Metadata DQ (Dynamic based on Column Comments!)
        validated_df = apply_metadata_dq(batch_df, target_table)
    else:
        # If table doesn't exist (First Run), skip DQ so we can create it
        print("First Run: Creating target table...")
        validated_df = batch_df.withColumn("is_invalid", lit(False)).withColumn("dq_errors", lit(""))
    
    # B. Split: Clean vs Quarantine
    clean_df = validated_df.filter("is_invalid = false").drop("is_invalid", "dq_errors")
    quarantine_df = validated_df.filter("is_invalid = true")
    
    # C. Save Quarantine
    if quarantine_df.count() > 0:
        print(f"⚠ {quarantine_df.count()} records quarantined.")
        quarantine_df.write.format("delta").mode("append").saveAsTable(quarantine_table)
    
    # D. Save Clean data (Upsert into Gold)
    if not table_exists:
        clean_df.write.format("delta").saveAsTable(target_table)
        return

    gold_table = DeltaTable.forName(spark, target_table)
    (gold_table.alias("t")
        .merge(clean_df.alias("s"), "t.customer_id = s.customer_id")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())

# 3. Execute Stream
print(f"Starting Comment-Driven Gold processing for {target_table}...")
query = (
    spark.readStream
    .table("workspace.silver.crm_customers_cdc")
    .writeStream
    .foreachBatch(process_gold_customers)
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .start()
)

query.awaitTermination()
print(f"✓ Gold processing complete. Total records in {target_table}: {spark.table(target_table).count():,}")